In [1]:
import sys
import os

# ensure local baseline package is importable
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))
from baseline.data import load_sequences, load_labels
from baseline.template_model import TemplateRepository
from baseline.search import seq_identity
from baseline.predict import generate_submission

sys.path.append(os.path.join(os.getcwd(), '..', 'scripts'))
from kabsch_utils import align_and_rmsd
from backbone_utils import extract_C1p_coords, extract_coords_from_submission

/Users/tatsuki/work/kaggle/kauto/.venv/lib/python3.13/site-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


In [2]:
data_dir = os.path.join('..', 'data', 'stanford-rna-3d-folding-2')
print('data_dir ->', data_dir)

test_seq = load_sequences(os.path.join(data_dir, 'validation_sequences.csv'))
train_seq = load_sequences(os.path.join(data_dir, 'train_sequences.csv'))
train_labels = load_labels(os.path.join(data_dir, 'train_labels.csv'))
print('loaded:', test_seq.shape, train_seq.shape, train_labels.shape)


data_dir -> ../data/stanford-rna-3d-folding-2
loaded: (28, 2) (5716, 2) (7794971, 9)


In [3]:
# Fit repository and generate a submission
repo = TemplateRepository()

repo.fit(train_seq, train_labels)
print('templates count:', len(repo.templates))

sub = generate_submission(test_seq, repo, seq_identity, n_structures=1)
print('submission rows:', len(sub))

sub.head()

templates count: 5716


KeyboardInterrupt: 

In [8]:
# Save submission
out_path = os.path.abspath(os.path.join(os.getcwd(),  'baseline_submission.csv'))
sub.to_csv(out_path, index=False)
print('saved ->', out_path)


saved -> /Users/tatsuki/work/kaggle/kauto/competitions/rna2/experiments/baseline_submission.csv


# Validation RMSD 結果
以下は `run_rmsd_evaluate.py` が出力した `validation_rmsd_per_target.csv` と `validation_rmsd_summary.json` の要約表示です。C1' 原子を用いた Kabsch+RMSD の結果を確認できます。

In [ ]:
# Interactive evaluation: provide a prediction DataFrame or CSV path and a ground-truth labels CSV or DataFrame
import os
import pandas as pd
import numpy as np

def evaluate_submission(pred_df=None, pred_csv=None, true_df=None, true_csv=None, min_pairs=3):
    """Evaluate a prediction against ground-truth labels.
    Returns (per_target_df, summary_dict).
    Supply either DataFrames or CSV paths.
    """
    if pred_df is None and pred_csv is None:
        raise ValueError('Provide pred_df or pred_csv')
    if true_df is None and true_csv is None:
        raise ValueError('Provide true_df or true_csv')
    if pred_df is None:
        pred_df = pd.read_csv(pred_csv)
    if true_df is None:
        true_df = pd.read_csv(true_csv)

    true_groups = extract_C1p_coords(true_df)
    pred_groups = extract_coords_from_submission(pred_df)
    results = []
    for tid, (true_coords, true_resids) in true_groups.items():
        pred_entry = pred_groups.get(tid)
        if pred_entry is None:
            results.append({'target_id': tid, 'n_matched': 0, 'mean_rmsd': None, 'median_rmsd': None, 'skipped': True})
            continue
        pred_coords, pred_resids = pred_entry
        true_map = {r: i for i,r in enumerate(true_resids)}
        pred_map = {r: i for i,r in enumerate(pred_resids)}
        common = sorted(set(true_map.keys()) & set(pred_map.keys()), key=lambda x:int(x) if x.isdigit() else x)
        pairs = []
        for r in common:
            t_idx = true_map[r]
            p_idx = pred_map[r]
            tc = true_coords[t_idx]
            pc = pred_coords[p_idx]
            if np.any(np.isnan(tc)) or np.any(np.isnan(pc)):
                continue
            pairs.append((tc, pc))
        if len(pairs) < min_pairs:
            results.append({'target_id': tid, 'n_matched': len(pairs), 'mean_rmsd': None, 'median_rmsd': None, 'skipped': True})
            continue
        P = np.vstack([t for t,p in pairs])
        Q = np.vstack([p for t,p in pairs])
        Q_aligned, rmsd = align_and_rmsd(P, Q)
        results.append({'target_id': tid, 'n_matched': len(pairs), 'mean_rmsd': float(rmsd), 'median_rmsd': float(np.median(np.linalg.norm(P-Q_aligned,axis=1))), 'skipped': False})
    df = pd.DataFrame(results)
    summary = {
        'n_targets': int(df.shape[0]),
        'n_evaluated': int((~df['skipped']).sum()),
        'mean_rmsd': None if df['mean_rmsd'].dropna().empty else float(df['mean_rmsd'].dropna().mean()),
        'median_rmsd': None if df['mean_rmsd'].dropna().empty else float(df['mean_rmsd'].dropna().median())
    }
    return df, summary

# Example usage: df,summary = evaluate_submission(pred_df=sub, true_csv=os.path.join('..','data','stanford-rna-3d-folding-2','validation_labels.csv'))
# display(df.head()); print(summary)

In [9]:
df, summary = evaluate_submission(pred_csv='./baseline_submission.csv', true_csv=os.path.join('..','data','stanford-rna-3d-folding-2','validation_labels.csv'))

In [12]:
import os, json
import pandas as pd

csv_path = os.path.join(os.getcwd(), 'validation_rmsd_per_target.csv')
json_path = os.path.join(os.getcwd(), 'validation_rmsd_summary.json')

if os.path.exists(json_path):
    with open(json_path) as f:
        summary = json.load(f)
    print('Validation summary:')
    print(summary)
else:
    print('Summary file not found:', json_path)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print('\nPer-target sample (first 10 rows):')
    display(df.head(10))
    print('\nTop 5 targets (lowest mean_rmsd):')
    display(df[df['skipped']==False].sort_values('mean_rmsd').head(5))
else:
    print('Per-target CSV not found:', csv_path)

Validation summary:
{'n_targets': 28, 'n_evaluated': 28, 'mean_rmsd': 34.47973280949871, 'median_rmsd': 31.896112632379058}

Per-target sample (first 10 rows):


,target_id,n_matched,mean_rmsd,median_rmsd,skipped
0,8ZNQ,28,15.373668,14.779075,False
1,9CFN,55,21.752716,19.057083,False
2,9E74,241,49.190164,46.648574,False
3,9E75,140,56.150164,50.836298,False
4,9E9Q,76,33.578116,31.142571,False
5,9EBP,76,28.124554,26.216915,False
6,9G4J,223,53.493649,46.990395,False
7,9G4P,66,16.106788,15.245878,False
8,9G4Q,76,31.455665,23.847245,False
9,9G4R,47,20.574248,18.205706,False



Top 5 targets (lowest mean_rmsd):


,target_id,n_matched,mean_rmsd,median_rmsd,skipped
24,9QZJ,19,12.732402,11.597557,False
10,9HRO,28,15.044589,14.617185,False
25,9RVP,25,15.221458,14.410096,False
0,8ZNQ,28,15.373668,14.779075,False
7,9G4P,66,16.106788,15.245878,False
